# KV Activation Visualization

기존 `run_model.py --collect_qkv`로 저장한 KV activation을 탐색하는 노트북입니다.

기본 저장 경로는 `activations/<model_dir>/*_{raw,rot}.pt`이며, tensor는 저장 직후 shape `(L, B, H, S, D)`를 `(L, H, B*S, D)`로 변환해 시각화합니다.

## 0. Activation 수집 예시

아직 activation 파일이 없다면 레포 루트에서 아래처럼 먼저 수집합니다.

```bash
python run_model.py --model meta-llama/Llama-2-7b-hf --collect_qkv --device cuda

# rotation 적용본도 같이 보고 싶으면
python run_model.py --model meta-llama/Llama-2-7b-hf --collect_qkv --rotate hadamard --device cuda
```

LLaMA 계열은 `k`, `v`, `k_rope`, `q_rope`, `k_grad`, `v_grad`가 저장될 수 있고, OPT 계열은 보통 `k`, `v`, `k_grad`, `v_grad`가 저장됩니다.

In [ ]:
from pathlib import Path
import math
import os

import torch
import numpy as np
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid", context="notebook")
except Exception:
    sns = None
    plt.style.use("default")

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

ACT_DIR = REPO_ROOT / "activations"
print(f"repo root: {REPO_ROOT}")
print(f"activation dir: {ACT_DIR}")

## 1. 설정

`MODEL_ID`는 Hugging Face model id 또는 저장 디렉터리명입니다. 예: `meta-llama/Llama-2-7b-hf`는 `meta-llama_Llama-2-7b-hf`로 저장됩니다.

In [ ]:
MODEL_ID = "meta-llama/Llama-2-7b-hf"
ROT_TAG = "raw"  # "raw" 또는 "rot"

# 자주 바꾸는 plot 대상
KIND = "k"       # "k", "v", "k_rope", "q_rope", "k_grad", "v_grad"
LAYER_IDX = 0
HEAD_IDX = 0

# 긴 sequence를 모두 그리면 느릴 수 있어 앞쪽 일부만 heatmap에 사용합니다.
MAX_TOKENS_FOR_HEATMAP = 256

model_dir = MODEL_ID.replace("/", "_")
act_root = ACT_DIR / model_dir
act_root

In [ ]:
def list_activation_sets(act_dir: Path = ACT_DIR):
    if not act_dir.exists():
        return []
    rows = []
    for model_path in sorted(p for p in act_dir.iterdir() if p.is_dir()):
        files = sorted(model_path.glob("*.pt"))
        tags = sorted({p.stem.rsplit("_", 1)[-1] for p in files if p.stem.endswith(("_raw", "_rot"))})
        kinds = sorted({p.stem.rsplit("_", 1)[0] for p in files if p.stem.endswith(("_raw", "_rot"))})
        rows.append({"model_dir": model_path.name, "tags": tags, "kinds": kinds, "n_files": len(files)})
    return rows

available = list_activation_sets()
if not available:
    print("No activation files found. Run `python run_model.py --collect_qkv ...` first.")
else:
    for row in available:
        print(f"{row['model_dir']} | tags={row['tags']} | kinds={row['kinds']} | files={row['n_files']}")

## 2. 로드 유틸리티

`run_model.py`가 저장한 `(L, B, H, S, D)` 텐서를 plotting에 편한 `(L, H, T, D)`로 변환합니다. 이미 `(L, H, T, D)`인 텐서도 그대로 처리합니다.

In [ ]:
def activation_path(kind: str, tag: str = ROT_TAG, root: Path = act_root) -> Path:
    return root / f"{kind}_{tag}.pt"

def to_layer_head_time_dim(x: torch.Tensor) -> torch.Tensor:
    """Normalize saved activations to (layers, heads, tokens, head_dim)."""
    if x.ndim == 5:
        # (L, B, H, S, D) -> (L, H, B*S, D)
        l, b, h, s, d = x.shape
        return x.transpose(1, 2).reshape(l, h, b * s, d)
    if x.ndim == 4:
        return x
    raise ValueError(f"Expected 4D or 5D tensor, got shape={tuple(x.shape)}")

def load_activation(kind: str, tag: str = ROT_TAG, root: Path = act_root) -> torch.Tensor:
    path = activation_path(kind, tag, root)
    if not path.exists():
        raise FileNotFoundError(f"Missing activation file: {path}")
    x = torch.load(path, map_location="cpu")
    x = to_layer_head_time_dim(x).float()
    print(f"loaded {path.name}: shape={tuple(x.shape)}, dtype={x.dtype}")
    return x

def available_kinds(tag: str = ROT_TAG, root: Path = act_root):
    if not root.exists():
        return []
    suffix = f"_{tag}.pt"
    return sorted(p.name[:-len(suffix)] for p in root.glob(f"*{suffix}"))

print("available kinds:", available_kinds())

In [ ]:
x = load_activation(KIND, ROT_TAG)
n_layers, n_heads, n_tokens, head_dim = x.shape
print({"layers": n_layers, "heads": n_heads, "tokens": n_tokens, "head_dim": head_dim})

## 3. 단일 Layer/Head Heatmap

행은 token/time 축, 열은 head dimension입니다. 값 범위가 큰 경우 percentile clipping을 적용해 패턴이 더 잘 보이게 합니다.

In [ ]:
def plot_head_heatmap(x: torch.Tensor, layer: int = LAYER_IDX, head: int = HEAD_IDX,
                      max_tokens: int = MAX_TOKENS_FOR_HEATMAP, clip_percentile: float = 99.0,
                      title: str | None = None):
    data = x[layer, head, :max_tokens].numpy()
    vmax = np.percentile(np.abs(data), clip_percentile)
    vmax = float(vmax) if vmax > 0 else None

    fig, ax = plt.subplots(figsize=(12, 5))
    im = ax.imshow(data, aspect="auto", cmap="coolwarm", vmin=None if vmax is None else -vmax, vmax=vmax)
    ax.set_xlabel("head dimension")
    ax.set_ylabel("token")
    ax.set_title(title or f"{KIND}_{ROT_TAG} | layer={layer}, head={head}")
    fig.colorbar(im, ax=ax, label="activation")
    plt.tight_layout()

plot_head_heatmap(x)

## 4. Layer/Head별 Activation 크기

각 layer/head에 대해 전체 token과 channel의 RMS 또는 max-abs를 요약합니다.

In [ ]:
def summarize_layer_head(x: torch.Tensor):
    rms = x.pow(2).mean(dim=(-1, -2)).sqrt()
    max_abs = x.abs().amax(dim=(-1, -2))
    mean_abs = x.abs().mean(dim=(-1, -2))
    return {"rms": rms, "max_abs": max_abs, "mean_abs": mean_abs}

summary = summarize_layer_head(x)

def plot_layer_head_metric(metric: torch.Tensor, name: str):
    fig, ax = plt.subplots(figsize=(12, 5))
    im = ax.imshow(metric.numpy(), aspect="auto", cmap="viridis")
    ax.set_xlabel("head")
    ax.set_ylabel("layer")
    ax.set_title(f"{KIND}_{ROT_TAG} {name} by layer/head")
    fig.colorbar(im, ax=ax, label=name)
    plt.tight_layout()

plot_layer_head_metric(summary["rms"], "RMS")
plot_layer_head_metric(summary["max_abs"], "max |x|")

In [ ]:
def plot_layer_metric_lines(summary: dict[str, torch.Tensor]):
    fig, ax = plt.subplots(figsize=(12, 4))
    for name, metric in summary.items():
        ax.plot(metric.mean(dim=1).numpy(), marker="o", label=f"mean head {name}")
    ax.set_xlabel("layer")
    ax.set_ylabel("value")
    ax.set_title(f"{KIND}_{ROT_TAG} layer-wise summary")
    ax.legend()
    plt.tight_layout()

plot_layer_metric_lines(summary)

## 5. Distribution / Outlier 확인

전체 tensor가 큰 경우 랜덤 샘플링해서 histogram을 그립니다.

In [ ]:
def sample_flat(x: torch.Tensor, n: int = 1_000_000, seed: int = 0):
    flat = x.flatten()
    if flat.numel() <= n:
        return flat.numpy()
    g = torch.Generator().manual_seed(seed)
    idx = torch.randperm(flat.numel(), generator=g)[:n]
    return flat[idx].numpy()

def plot_distribution(x: torch.Tensor, title: str | None = None, bins: int = 200):
    data = sample_flat(x)
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].hist(data, bins=bins, density=True, color="#4c78a8", alpha=0.85)
    axes[0].set_title("activation")
    axes[0].set_xlabel("value")
    axes[0].set_ylabel("density")

    abs_data = np.abs(data)
    axes[1].hist(abs_data, bins=bins, density=True, color="#f58518", alpha=0.85)
    axes[1].set_yscale("log")
    axes[1].set_title("absolute activation, log density")
    axes[1].set_xlabel("|value|")
    fig.suptitle(title or f"{KIND}_{ROT_TAG} sampled distribution")
    plt.tight_layout()

plot_distribution(x)

In [ ]:
def print_quantiles(x: torch.Tensor, name: str = "activation"):
    vals = x.abs().flatten()
    qs = torch.tensor([0.5, 0.9, 0.95, 0.99, 0.999], dtype=torch.float32)
    qv = torch.quantile(vals, qs)
    print(name)
    for q, v in zip(qs.tolist(), qv.tolist()):
        print(f"  p{q * 100:05.2f}: {v:.6g}")
    print(f"  max  : {vals.max().item():.6g}")

print_quantiles(x, f"{KIND}_{ROT_TAG}")

## 6. Token-wise Norm 변화

특정 layer/head에서 token 위치별 norm 변화를 봅니다. 긴 context에서 특정 구간에 outlier가 몰리는지 확인할 때 유용합니다.

In [ ]:
def plot_token_norm(x: torch.Tensor, layer: int = LAYER_IDX, head: int = HEAD_IDX):
    data = x[layer, head]
    l2 = data.pow(2).sum(dim=-1).sqrt().numpy()
    max_abs = data.abs().amax(dim=-1).numpy()

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(l2, label="L2 norm")
    ax.plot(max_abs, label="max |x|", alpha=0.8)
    ax.set_xlabel("token")
    ax.set_title(f"{KIND}_{ROT_TAG} token-wise norm | layer={layer}, head={head}")
    ax.legend()
    plt.tight_layout()

plot_token_norm(x)

## 7. Raw vs Rot 비교

`raw`와 `rot` activation이 둘 다 있으면 layer/head별 RMS와 분포를 비교합니다.

In [ ]:
def maybe_load_pair(kind: str, root: Path = act_root):
    raw_path = activation_path(kind, "raw", root)
    rot_path = activation_path(kind, "rot", root)
    if not raw_path.exists() or not rot_path.exists():
        print(f"Need both {raw_path.name} and {rot_path.name} for comparison.")
        return None, None
    return load_activation(kind, "raw", root), load_activation(kind, "rot", root)

raw_x, rot_x = maybe_load_pair(KIND)

if raw_x is not None:
    raw_rms = summarize_layer_head(raw_x)["rms"]
    rot_rms = summarize_layer_head(rot_x)["rms"]
    ratio = rot_rms / raw_rms.clamp_min(1e-12)

    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    for ax, metric, title in zip(axes, [raw_rms, rot_rms, ratio], ["raw RMS", "rot RMS", "rot/raw RMS"]):
        im = ax.imshow(metric.numpy(), aspect="auto", cmap="viridis")
        ax.set_xlabel("head")
        ax.set_ylabel("layer")
        ax.set_title(title)
        fig.colorbar(im, ax=ax)
    plt.tight_layout()

    print_quantiles(raw_x, f"{KIND}_raw")
    print_quantiles(rot_x, f"{KIND}_rot")

## 8. K/V 한 번에 비교

`k`와 `v`를 같은 tag에서 비교합니다.

In [ ]:
def plot_kv_layer_summary(tag: str = ROT_TAG):
    tensors = {}
    for kind in ["k", "v", "k_rope"]:
        path = activation_path(kind, tag)
        if path.exists():
            tensors[kind] = load_activation(kind, tag)

    if not tensors:
        print(f"No k/v activation files found for tag={tag}")
        return

    fig, ax = plt.subplots(figsize=(12, 4))
    for kind, tensor in tensors.items():
        rms_by_layer = summarize_layer_head(tensor)["rms"].mean(dim=1).numpy()
        ax.plot(rms_by_layer, marker="o", label=kind)
    ax.set_xlabel("layer")
    ax.set_ylabel("mean RMS across heads")
    ax.set_title(f"KV activation RMS by layer ({tag})")
    ax.legend()
    plt.tight_layout()

plot_kv_layer_summary(ROT_TAG)